In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import requests
from google.colab import userdata
import os

In [3]:
base = '/content/drive/MyDrive/Deforestation_Project'
os.makedirs(base, exist_ok=True)
folders = ["data/S1/raw", "data/S1/processed", "data/S2/raw", "data/S2/processed"]
for fol in folders:
  os.makedirs(os.path.join(base, fol), exist_ok=True)


In [13]:
# Coordiantes of Study area Rondonia

STUDY_AREA = "Rondonia, Brazil"
print(f"The study are of this work is {STUDY_AREA}")

bbox_co = {
    "min_lon": -62.9,
    "min_lat": -11.7,
    "max_lon": -62.1,
    "max_lat": -11.0
}

print("Cordinates=", bbox_co)
# print("min_lon=", bbox_co["min_lon"])

footprint = f"POLYGON(({bbox_co['min_lon']} {bbox_co['min_lat']}, {bbox_co['min_lon']} {bbox_co['max_lat']}, {bbox_co['max_lon']} {bbox_co['max_lat']}, {bbox_co['max_lon']} {bbox_co['min_lat']}, {bbox_co['min_lon']} {bbox_co['min_lat']}))"

The study are of this work is Rondonia, Brazil
Cordinates= {'min_lon': -62.9, 'min_lat': -11.7, 'max_lon': -62.1, 'max_lat': -11.0}


In [14]:
# Connection to Copernicus server for access token

def get_token():

  username = userdata.get('CDSE_USER')
  password = userdata.get('CDSE_PASS')
  data = {"client_id": "cdse-public", "username": username, "password": password, "grant_type": "password"}
  url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'          # url for access token

  response = requests.post(url, data = data)
  print(response.status_code)
  token = response.json()['access_token']

  return token

In [15]:
token = get_token()
print(token[:20])

200
eyJhbGciOiJSUzI1NiIs


In [16]:
# Complete filters for OData (Open Data Protocol) based search for scenes or tiles

base_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter="

## Common filters
date_filter = "ContentDate/Start gt 2023-01-01T00:00:00.000Z and ContentDate/Start lt 2023-12-31T00:00:00.000Z"
area_filter = f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint}')"
limit = "&$top=200"

## SENTINEL 1 FILTERES
collection_filter_s1 = "Collection/Name eq 'SENTINEL-1'"
product_type_s1 = "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'GRD')"   #Ground Range Detected should be used(contains only amplitude)
orbit_dir = "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'orbitDirection' and att/OData.CSC.StringAttribute/Value eq 'DESCENDING')"

## SENTINEL 2 FILTERS
tile_filter = "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'tileId' and att/OData.CSC.StringAttribute/Value eq '20LNN')"   #SINCE WHOLE STUDY AREA WAS FOUND TO BE INSIDE A TILE ITS FILTERED OUT AND DOWNLOADED
collection_filter_s2 = "Collection/Name eq 'SENTINEL-2'"
product_type_s2 = "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A')"
cloud_cover = "Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt 30.00)"

**Sentinel 1 filter explanation**:
The product type is GRD — Ground Range Detected. Specifically IW_GRDH Interferometric Wide swath, High resolution.
It contains amplitude only, no phase
Simpler to work with than SLC
Standard format for land cover analysis
Already multi-looked to reduce speckle

Sentinel-1 looks to the right of its flight direction. On a descending pass (north to south) over South America, looking right means looking west, giving a good viewing geometry over the Amazon region.
More practically the descending orbit has more consistent historical coverage over Rondônia than ascending. Mixing both orbits adds noise to your time series because the viewing angle differs, making backscatter values not directly comparable.
The filter value is DESCENDING

**S1**: SINCE WHOLE STUDY AREA WAS FOUND TO BE INSIDE A TILE ITS FILTERED OUT AND DOWNLOADED

In [17]:
# Search URL for S1

SEARCH_FILTER_S1 = base_url + collection_filter_s1 + " and " + product_type_s1 + " and " + orbit_dir + " and " + area_filter + " and " + date_filter + limit
print(SEARCH_FILTER_S1)

# the access token is in the headers
headers = {"Authorization": f"Bearer {token}"}

# The requested scenes will be stored in search response
search_response_s1 = requests.get(SEARCH_FILTER_S1, headers = headers)
print("Status code for S1 search:",search_response_s1.status_code)

# Data of the available the scenes can be accessed using json()
results_s1 = search_response_s1.json()
# print("Results s1:", results_s1)

# value holds the scene information
scenes_s1 = results_s1["value"]

#Total no of scenes
print("Total S1 scenes:", len(scenes_s1))

#First Scene
print("First S1 scene name:", scenes_s1[0]["Name"])

# tile = set()
# for scene in scenes_s1:
#   print(scene["Name"][11:19])      # Check on acqusition dates
#   print(scene["Name"][38:44])      # Check on tile names
#   tile.add(scene["Name"][38:44])   # scenes are stored to set just to check for duplicates since we only use a single tile
#     print(tile)


https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name eq 'SENTINEL-1' and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'GRD') and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'orbitDirection' and att/OData.CSC.StringAttribute/Value eq 'DESCENDING') and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON((-62.9 -11.7, -62.9 -11.0, -62.1 -11.0, -62.1 -11.7, -62.9 -11.7))') and ContentDate/Start gt 2023-01-01T00:00:00.000Z and ContentDate/Start lt 2023-12-31T00:00:00.000Z&$top=200
Status code for S1 search: 200
Total S1 scenes: 51
First S1 scene name: S1A_IW_GRDH_1SDV_20230708T095005_20230708T095030_049330_05EE9A_9CD7.SAFE


In [18]:
# Search URL for S2

SEARCH_FILTER_S2 = base_url + collection_filter_s2 + " and " + product_type_s2 + " and " + tile_filter + " and " + cloud_cover + " and " + area_filter + " and " + date_filter + limit
print(SEARCH_FILTER_S2)

# the access token is in the headers
headers = {"Authorization": f"Bearer {token}"}

# The requested scenes will be stored in search response
search_response_s2 = requests.get(SEARCH_FILTER_S2, headers = headers)
print(search_response_s2.status_code)

# Data of the available the scenes can be accessed using json()
results_s2 = search_response_s2.json()
# print(results_s2)

# value holds the scene information
scenes_s2 = results_s2["value"]

#Total no of scenes
print(len(scenes_s2))

#First Scene
print(scenes_s2[0]["Name"])

# tile = set()
# for scene in scenes_s1:
#   print(scene["Name"][11:19])      # Check on acqusition dates
#   print(scene["Name"][38:44])      # Check on tile names
#   tile.add(scene["Name"][38:44])   # scenes are stored to set just to check for duplicates since we only use a single tile
#     print(tile)


https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name eq 'SENTINEL-2' and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'tileId' and att/OData.CSC.StringAttribute/Value eq '20LNN') and Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt 30.00) and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON((-62.9 -11.7, -62.9 -11.0, -62.1 -11.0, -62.1 -11.7, -62.9 -11.7))') and ContentDate/Start gt 2023-01-01T00:00:00.000Z and ContentDate/Start lt 2023-12-31T00:00:00.000Z&$top=200
200
67
S2A_MSIL2A_20230326T142711_N0510_R053_T20LNN_20240826T091017.SAFE


In [20]:
# Sentienl 2 Downloading

# first = scenes_s2[:2]    # just to check the downloads for first 2 scene. Change the scenes in below loop to 'first'

for i, tiles in enumerate(scenes_s2):                                  # the tokens are refreshed after 5 downloads so it wont get canceled in between
  if i % 5 == 0:
    token = get_token()
  headers = {"Authorization": f"Bearer {token}"}
  ids = tiles["Id"]
  url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({ids})/$value"
  filename = tiles["Name"] + ".zip"
  filepath = os.path.join(base,  folders[2], filename)
  if os.path.exists(filepath):
    print(f"{tiles['Name']}, Already Downloaded")
    continue
  down_req = requests.get(url, headers = headers, allow_redirects=True, stream = True)
  if down_req.status_code == 200:
    with open(filepath, "wb") as f:
      for chunk in down_req.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
    print(f"{tiles['Name']}, Downloaded")
  else:
      print(f"Failed: {tiles['Name']}, status: {down_req.status_code}")

print("All scenes from SENTINEL-2 downloaded")


200
S2A_MSIL2A_20230326T142711_N0510_R053_T20LNN_20240826T091017.SAFE, Already Downloaded
S2B_MSIL2A_20230328T141719_N0510_R010_T20LNN_20240830T005625.SAFE, Already Downloaded
S2B_MSIL2A_20230206T141709_N0510_R010_T20LNN_20240815T024904.SAFE, Already Downloaded
S2B_MSIL2A_20230301T142709_N0510_R053_T20LNN_20240817T121956.SAFE, Already Downloaded
S2B_MSIL2A_20230226T141709_N0510_R010_T20LNN_20240818T192152.SAFE, Already Downloaded
200
S2B_MSIL2A_20230828T142719_N0510_R053_T20LNN_20241019T002816.SAFE, Already Downloaded
S2B_MSIL2A_20230818T142719_N0510_R053_T20LNN_20241025T214836.SAFE, Already Downloaded
S2B_MSIL2A_20230726T141719_N0510_R010_T20LNN_20240929T133606.SAFE, Already Downloaded
S2B_MSIL2A_20230917T142719_N0510_R053_T20LNN_20241027T080145.SAFE, Already Downloaded
S2A_MSIL2A_20230922T142711_N0510_R053_T20LNN_20241026T093629.SAFE, Already Downloaded
200
S2A_MSIL2A_20230714T142721_N0510_R053_T20LNN_20240913T150750.SAFE, Already Downloaded
S2A_MSIL2A_20230704T142721_N0510_R053_T20L

In [20]:
# Sentienl 1 Downloading

# first = scenes_s1[:2]    # just to check the downloads for first 2 scene. Change the scenes in below loop to 'first'

for i, tiles in enumerate(scenes_s1):
  if i % 5 == 0:                                                       # the tokens are refreshed after 5 downloads so it wont get canceled in between
    token = get_token()
  headers = {"Authorization": f"Bearer {token}"}
  ids = tiles["Id"]
  url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({ids})/$value"
  filename = tiles["Name"] + ".zip"
  filepath = os.path.join(base,  folders[0], filename)
  if os.path.exists(filepath):
    print(f"{tiles['Name']}, Already Downloaded")
    continue
  down_req = requests.get(url, headers = headers, allow_redirects=True, stream = True)
  if down_req.status_code == 200:
    with open(filepath, "wb") as f:
      for chunk in down_req.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
    print(f"{tiles['Name']}, Downloaded")
  else:
      print(f"Failed: {tiles['Name']}, status: {down_req.status_code}")

print("All scenes SENTINEL-1 downloaded")


200
S1A_IW_GRDH_1SDV_20230708T095005_20230708T095030_049330_05EE9A_9CD7.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20230214T095000_20230214T095025_047230_05AAD4_013E.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20230415T095001_20230415T095026_048105_05C876_4543.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20230504T094203_20230504T094228_048382_05D1BE_8EBE.SAFE, Already Downloaded
S1A_IW_GRDH_1SDV_20230801T095006_20230801T095031_049680_05F959_0CC1.SAFE, Downloaded
200
S1A_IW_GRDH_1SDV_20230901T094210_20230901T094235_050132_060889_0A18.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20230925T094211_20230925T094236_050482_061475_5E63.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20231007T094211_20231007T094236_050657_061A75_A7AA.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20231230T094206_20231230T094231_051882_0644A5_E1D1.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20230310T095000_20230310T095025_047580_05B6BD_4C23.SAFE, Downloaded
200
S1A_IW_GRDH_1SDV_20230305T094202_20230305T094227_047507_05B44E_2EFF.SAFE, Downloaded
S1A_IW_GRDH_1SDV_20230221T094201_20230221T094